# 06 · Compound-level interaction summaries

Normalize three planned interaction classes for comparison: hydrophobic contacts, hydrogen bonds, and ionic contacts. Other extracted classes remain available in notebook 05. All metadata rows are retained, including zero reported contacts.

## Configuration

Use `config.local.json` as described in [the workflow guide](../docs/WORKFLOW.md). Inputs must be authorized for local analysis. Outputs are cleared in version control; no research result is asserted by this notebook.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis/interactions.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Start Jupyter from this repository')
sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.notebook_setup import load_config, load_manifest
config = load_config()
out = config['output_dir']


## Map event names and preserve keys

In [ ]:
events = pd.read_csv(out / 'interaction_events.csv', dtype={'compound_id': str, 'pose_id': str})
metadata = pd.read_csv(out / 'pose_metadata.csv', dtype={'compound_id': str, 'pose_id': str})
from analysis.interactions import FEATURES, load_counts, summarize
mapping = {'hydrophobic': 'hydrophobic', 'hbond_ligand_acceptor': 'hbond',
           'hbond_ligand_donnor': 'hbond', 'hbond_ligand_donor': 'hbond',
           'ionic_ligand_positive': 'ionic', 'ionic_ligand_negative': 'ionic'}
events['analysis_feature'] = events.feature.map(mapping)
selected = events.dropna(subset=['analysis_feature'])
counts = selected.groupby(['compound_id', 'pose_id', 'group', 'analysis_feature']).size().unstack(fill_value=0)
counts = counts.reindex(columns=FEATURES, fill_value=0).reset_index()
counts = metadata.merge(counts, on=['compound_id', 'pose_id', 'group'], how='left', validate='one_to_one')
counts[FEATURES] = counts[FEATURES].fillna(0).astype(int)
counts.to_csv(out / 'pose_counts.csv', index=False)
validated = load_counts(out / 'pose_counts.csv')
means, presence = summarize(validated)
means.to_csv(out / 'compound_mean_counts.csv', index=False)
presence.to_csv(out / 'compound_presence.csv', index=False)
means.head()

## Plot compound-level distributions

In [ ]:
import seaborn as sns
long = means.melt(id_vars=['compound_id', 'group'], value_vars=FEATURES,
                  var_name='feature', value_name='mean_contacts')
grid = sns.catplot(data=long, x='group', y='mean_contacts', col='feature', kind='box', color='#9db9cd')
grid.set_axis_labels('Compound group', 'Mean contacts across poses')
grid.figure.savefig(out / 'compound_distributions.svg', bbox_inches='tight')
plt.show()